In [1]:
!pip install ortools torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.

In [2]:
# import os
# import re
# import time
# import numpy as np
# import pandas as pd
# import torch
# from ortools.linear_solver import pywraplp
# from torch_geometric.data import Data
# from joblib import Parallel, delayed

# # ==============================================================================
# # DATASET LOADING & PREPROCESSING
# # ==============================================================================
# def load_all_datasets(folder_path):
#     distance_matrices = {}
#     demand_matrices = {}
    
#     if not os.path.exists(folder_path):
#         print(f"Error: The directory '{folder_path}' does not exist.")
#         return None, None

#     print(f"Scanning directory: {folder_path}...\n")
#     for filename in os.listdir(folder_path):
#         file_path = os.path.join(folder_path, filename)
#         if not os.path.isfile(file_path): continue
            
#         match = re.search(r'\d+', filename)
#         if not match: continue
#         nodes = int(match.group())
        
#         try:
#             if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
#                 df = pd.read_csv(file_path, header=None)
#                 distance_matrices[nodes] = df.values
#                 print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes.")
                
#             elif filename.startswith('wij'):
#                 if filename.endswith('.xlsx'): df = pd.read_excel(file_path)
#                 elif filename.endswith('.csv'): df = pd.read_csv(file_path)
#                 else: continue
                
#                 if df.shape[1] > nodes: df = df.iloc[:, 1:]
#                 demand_matrices[nodes] = df.values
#                 print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes.")
#         except Exception as e:
#             pass
#     return distance_matrices, demand_matrices

# def generate_demand_scenarios(W_matrix, num_scenarios=50, variance_factor=0.2, seed=42):
#     """Generates 50 robust scenarios to train the AI on the true CVaR landscape"""
#     np.random.seed(seed)
#     n = W_matrix.shape[0]
#     W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
#     W_scenarios = np.maximum(W_scenarios, 0)
#     return np.transpose(W_scenarios, (1, 2, 0))

# def generate_subsampled_training_data(real_distances, real_demands, target_N):
#     N = len(real_distances)
#     if target_N >= N: return real_distances, real_demands
#     indices = np.random.choice(N, target_N, replace=False)
#     return real_distances[indices][:, indices], real_demands[indices][:, indices]

# def calculate_node_production_attraction(W_matrix):
#     return np.sum(W_matrix, axis=1), np.sum(W_matrix, axis=0)

# # ==============================================================================
# # EXACT SOLVER (NO TIME LIMIT)
# # ==============================================================================
# def solve_exact_robust_model_no_limit(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9):
#     """
#     Solves the Min-Max Robust CVaR model with NO TIME LIMIT.
#     This guarantees 100% mathematical optimality for the training labels.
#     """
#     N = len(distances)
#     num_scenarios = W_scenarios.shape[2]
    
#     dist_scale = np.max(distances) if np.max(distances) > 0 else 1.0
#     w_scale = np.max(W_scenarios) if np.max(W_scenarios) > 0 else 1.0
#     scaled_distances, scaled_W = distances / dist_scale, W_scenarios / w_scale
#     cost_multiplier = dist_scale * w_scale 
    
#     solver = pywraplp.Solver.CreateSolver('SCIP')
#     if not solver: return None, float('inf')
    
#     # NOTE: solver.SetTimeLimit() has been intentionally removed!
#     # The solver will run until the mathematical proof of optimality is 100% complete.

#     y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(N)}
#     z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for k in range(N) for i in range(N)}
#     mu = solver.NumVar(0, solver.infinity(), 'mu')
#     lambda_s = {s: solver.NumVar(0, solver.infinity(), f'lambda_{s}') for s in range(num_scenarios)}

#     solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)

#     for i in range(N):
#         solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)
#         for k in range(N):
#             solver.Add(z[i, k] <= y[k])

#     for s in range(num_scenarios):
#         W_s = scaled_W[:, :, s]
#         cost_s_expr = 0
#         for i in range(N):
#             for k in range(N):
#                 cost_s_expr += sum(W_s[i, j] for j in range(N)) * scaled_distances[i, k] * z[i, k]
#                 cost_s_expr += sum(W_s[j, i] for j in range(N)) * scaled_distances[i, k] * z[i, k]
#                 cost_s_expr += sum(W_s[i, j] for j in range(N)) * (alpha * scaled_distances[k, k]) * z[i, k]
#         solver.Add(lambda_s[s] >= cost_s_expr - mu)

#     penalty_weight = 1.0 / (num_scenarios * (1.0 - beta))
#     solver.Minimize(mu + penalty_weight * solver.Sum([lambda_s[s] for s in range(num_scenarios)]))

#     status = solver.Solve()
#     if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
#         return [k for k in range(N) if y[k].solution_value() > 0.5], solver.Objective().Value() * cost_multiplier
#     return None, float('inf')

# # ==============================================================================
# # PYTORCH DATA FORMATTING
# # ==============================================================================
# def create_multi_graph_data(distances, demands, labels):
#     N = len(distances)
#     production, attraction = calculate_node_production_attraction(demands)
#     spatial_centrality = 1.0 / (np.sum(distances, axis=1) + 1e-6)
    
#     x = torch.tensor(np.column_stack((
#         production / (np.max(production) or 1.0), 
#         attraction / (np.max(attraction) or 1.0),
#         spatial_centrality / (np.max(spatial_centrality) or 1.0)
#     )), dtype=torch.float)

#     edge_index, edge_attr = [], []
#     dist_max, dem_max = (np.max(distances) or 1.0), (np.max(demands) or 1.0)
#     flow = demands + demands.T
#     flow_max = np.max(flow) or 1.0

#     for i in range(N):
#         for j in range(N):
#             if i != j:
#                 edge_index.append([i, j])
#                 edge_attr.append([distances[i, j]/dist_max, demands[i, j]/dem_max, flow[i, j]/flow_max])

#     data = Data(x=x, 
#                 edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
#                 edge_attr=torch.tensor(edge_attr, dtype=torch.float),
#                 y=torch.tensor(labels, dtype=torch.float))
#     return data

# # ==============================================================================
# # PARALLEL WORKER FUNCTION
# # ==============================================================================
# def _generate_offline_worker(seed_offset, target_N, p_hubs, common_nodes, distance_matrices, demand_matrices):
#     """Independent worker to generate and solve a graph for a specific N and p."""
#     # Ensure unique seed for every parallel worker
#     np.random.seed((int(time.time() * 1000) + seed_offset) % 123456789)
    
#     valid_bases = [n for n in common_nodes if n >= target_N]
#     if not valid_bases:
#         return None
        
#     base_N = np.random.choice(valid_bases)
#     train_dist, train_dem = generate_subsampled_training_data(
#         distance_matrices[base_N], demand_matrices[base_N], target_N
#     )
    
#     W_scen = generate_demand_scenarios(train_dem, num_scenarios=100, seed=np.random.randint(10000)) 
    
#     # --- START NO-LIMIT TIMING ---
#     start_time = time.time()
#     exact_hubs, optimal_cost = solve_exact_robust_model_no_limit(train_dist, W_scen, p_hubs=p_hubs)
#     elapsed_time = time.time() - start_time
#     # --- END TIMING ---
    
#     if exact_hubs:
#         labels = np.zeros(target_N)
#         labels[exact_hubs] = 1.0
#         graph_data = create_multi_graph_data(train_dist, train_dem, labels)
        
#         print(f"   [✓] N={target_N:3d} | p={p_hubs} | Time to Prove Optimality: {elapsed_time:7.2f}s | Optimal Cost: {optimal_cost:.2e}")
#         return graph_data
#     else:
#         print(f"   [X] N={target_N:3d} | p={p_hubs} | Failed to find feasible solution.")
#         return None

# # ==============================================================================
# # MASTER GENERATION LOOP
# # ==============================================================================
# if __name__ == "__main__":
#     print("==================================================")
#     print("🛠️ STARTING PARALLEL OFFLINE DATASET GENERATOR")
#     print("==================================================")
    
#     DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
#     distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
#     if distance_matrices and demand_matrices:
#         common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
#         largest_available_N = max(common_nodes)
        
#         # Matrix of combinations we want to generate
#         target_sizes_to_generate = [20, 21, 22, 23, 24, 25, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100, 125, 150, 175]
#         p_values_to_generate = [2, 3, 4, 5]
#         graphs_per_combo = 7 # e.g. 7 graphs * 20 sizes * 4 p_values = 560 Total Graphs
        
#         tasks = []
#         seed_offset = 0
        
#         for target_N in target_sizes_to_generate:
#             if target_N > largest_available_N:
#                 continue
#             for p in p_values_to_generate:
#                 for _ in range(graphs_per_combo):
#                     tasks.append((seed_offset, target_N, p))
#                     seed_offset += 1
        
#         print(f"\n[Data Generation] Starting limitless SCIP evaluations for {len(tasks)} total graphs across all CPU cores...")
        
#         # Parallel Execution
#         # joblib automatically captures print statements from workers and prints them sequentially so the log stays readable
#         parallel_results = Parallel(n_jobs=-1)(
#             delayed(_generate_offline_worker)(
#                 seed, n, p, common_nodes, distance_matrices, demand_matrices
#             ) for seed, n, p in tasks
#         )
        
#         # Filter out failed runs
#         training_dataset = [g for g in parallel_results if g is not None]

#         # Save the massive dataset to the disk
#         output_file = "offline_training_dataset.pt"
#         torch.save(training_dataset, output_file)
        
#         print("\n==================================================")
#         print(f"✅ GENERATION COMPLETE!")
#         print(f"💾 Saved {len(training_dataset)} perfectly optimal graphs to '{output_file}'")
#         print("   You can now load this file instantly in your main pipeline using: ")
#         print("   training_graphs = torch.load('offline_training_dataset.pt')")
#         print("==================================================")
#     else:
#         print("\n[!] Directory not found or empty.")

In [3]:
# import os
# import re
# import time
# import copy
# import random
# import numpy as np
# import pandas as pd
# import itertools
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.cluster import KMeans
# from ortools.linear_solver import pywraplp
# from joblib import Parallel, delayed

# import torch
# import torch.nn.functional as F
# from torch_geometric.data import Data
# from torch_geometric.nn import TransformerConv

# # ==============================================================================
# # DATASET LOADING & PREPROCESSING
# # ==============================================================================
# def load_all_datasets(folder_path):
#     distance_matrices = {}
#     demand_matrices = {}
    
#     if not os.path.exists(folder_path):
#         print(f"Error: The directory '{folder_path}' does not exist.")
#         return None, None

#     print(f"Scanning directory: {folder_path}...\n")
#     for filename in os.listdir(folder_path):
#         file_path = os.path.join(folder_path, filename)
#         if not os.path.isfile(file_path): continue
            
#         match = re.search(r'\d+', filename)
#         if not match: continue
#         nodes = int(match.group())
        
#         try:
#             if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
#                 df = pd.read_csv(file_path, header=None)
#                 distance_matrices[nodes] = df.values
#                 print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes.")
                
#             elif filename.startswith('wij'):
#                 if filename.endswith('.xlsx'): df = pd.read_excel(file_path)
#                 elif filename.endswith('.csv'): df = pd.read_csv(file_path)
#                 else: continue
                
#                 if df.shape[1] > nodes: df = df.iloc[:, 1:]
#                 demand_matrices[nodes] = df.values
#                 print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes.")
#         except Exception as e:
#             pass
#     return distance_matrices, demand_matrices

# def calculate_node_production_attraction(W_matrix):
#     return np.sum(W_matrix, axis=1), np.sum(W_matrix, axis=0)

# def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
#     np.random.seed(seed)
#     n = W_matrix.shape[0]
#     W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
#     W_scenarios = np.maximum(W_scenarios, 0)
#     return np.transpose(W_scenarios, (1, 2, 0))

# def calculate_path_costs(C_matrix, alpha=0.5):
#     C_ik = C_matrix[:, np.newaxis, :, np.newaxis]
#     C_km = C_matrix[np.newaxis, np.newaxis, :, :]
#     C_mj = C_matrix.T[np.newaxis, :, np.newaxis, :]
#     return C_ik + (alpha * C_km) + C_mj

# # ==============================================================================
# # EXACT SOLVER & MATHEMATICAL EVALUATION
# # ==============================================================================
# def solve_exact_robust_model(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9, time_limit_mins=1.5, fixed_hubs=None):
#     N = len(distances)
#     num_scenarios = W_scenarios.shape[2]
    
#     dist_scale = np.max(distances) if np.max(distances) > 0 else 1.0
#     w_scale = np.max(W_scenarios) if np.max(W_scenarios) > 0 else 1.0
#     scaled_distances, scaled_W = distances / dist_scale, W_scenarios / w_scale
#     cost_multiplier = dist_scale * w_scale 
    
#     solver = pywraplp.Solver.CreateSolver('SCIP')
#     if not solver: return None, float('inf')
#     solver.SetTimeLimit(int(time_limit_mins * 60 * 1000))

#     y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(N)}
#     z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for k in range(N) for i in range(N)}
#     mu = solver.NumVar(0, solver.infinity(), 'mu')
#     lambda_s = {s: solver.NumVar(0, solver.infinity(), f'lambda_{s}') for s in range(num_scenarios)}

#     solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)

#     if fixed_hubs is not None:
#         for k in range(N):
#             solver.Add(y[k] == (1 if k in fixed_hubs else 0))

#     for i in range(N):
#         solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)
#         for k in range(N):
#             solver.Add(z[i, k] <= y[k])

#     for s in range(num_scenarios):
#         W_s = scaled_W[:, :, s]
#         cost_s_expr = 0
#         for i in range(N):
#             for k in range(N):
#                 cost_s_expr += sum(W_s[i, j] for j in range(N)) * scaled_distances[i, k] * z[i, k]
#                 cost_s_expr += sum(W_s[j, i] for j in range(N)) * scaled_distances[i, k] * z[i, k]
#                 cost_s_expr += sum(W_s[i, j] for j in range(N)) * (alpha * scaled_distances[k, k]) * z[i, k]
#         solver.Add(lambda_s[s] >= cost_s_expr - mu)

#     penalty_weight = 1.0 / (num_scenarios * (1.0 - beta))
#     solver.Minimize(mu + penalty_weight * solver.Sum([lambda_s[s] for s in range(num_scenarios)]))

#     status = solver.Solve()
#     if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
#         return [k for k in range(N) if y[k].solution_value() > 0.5], solver.Objective().Value() * cost_multiplier
#     return None, float('inf')

# def evaluate_heuristic_robust_cost(distances, W_scenarios, hubs, alpha=0.5, beta=0.9):
#     N, num_scenarios = len(distances), W_scenarios.shape[2]
#     allocation = {i: min(hubs, key=lambda h: distances[i, h]) for i in range(N)}
    
#     path_costs = np.zeros((N, N))
#     for i in range(N):
#         for j in range(N):
#             hi, hj = allocation[i], allocation[j]
#             path_costs[i, j] = distances[i, hi] + alpha * distances[hi, hj] + distances[hj, j]
            
#     scenario_costs = np.tensordot(path_costs, W_scenarios, axes=([0, 1], [0, 1]))
#     sorted_costs = np.sort(scenario_costs)
#     var_index = int(beta * num_scenarios)
#     mu = sorted_costs[var_index]
    
#     return mu + np.sum(sorted_costs[var_index:] - mu) / (num_scenarios * (1.0 - beta))

# # ==============================================================================
# # GRAPH NEURAL NETWORK & HYPERPARAMETER TUNING
# # ==============================================================================
# def create_multi_graph_data(distances, demands, labels=None):
#     N = len(distances)
#     production, attraction = calculate_node_production_attraction(demands)
#     spatial_centrality = 1.0 / (np.sum(distances, axis=1) + 1e-6)
    
#     x = torch.tensor(np.column_stack((
#         production / (np.max(production) or 1.0), 
#         attraction / (np.max(attraction) or 1.0),
#         spatial_centrality / (np.max(spatial_centrality) or 1.0)
#     )), dtype=torch.float)

#     edge_index, edge_attr = [], []
#     dist_max, dem_max = (np.max(distances) or 1.0), (np.max(demands) or 1.0)
#     flow = demands + demands.T
#     flow_max = np.max(flow) or 1.0

#     for i in range(N):
#         for j in range(N):
#             if i != j:
#                 edge_index.append([i, j])
#                 edge_attr.append([distances[i, j]/dist_max, demands[i, j]/dem_max, flow[i, j]/flow_max])

#     data = Data(x=x, 
#                 edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
#                 edge_attr=torch.tensor(edge_attr, dtype=torch.float))
#     if labels is not None: data.y = torch.tensor(labels, dtype=torch.float)
#     return data

# class MultiGraphHubRanker(torch.nn.Module):
#     def __init__(self, node_in_dim=3, edge_in_dim=3, hidden_dim=64, num_layers=2):
#         super(MultiGraphHubRanker, self).__init__()
#         self.num_layers = num_layers
#         self.convs = torch.nn.ModuleList()
        
#         self.convs.append(TransformerConv(node_in_dim, hidden_dim, edge_dim=edge_in_dim))
#         for _ in range(num_layers - 1):
#             self.convs.append(TransformerConv(hidden_dim, hidden_dim, edge_dim=edge_in_dim))
            
#         self.fc1 = torch.nn.Linear(hidden_dim, 32)
#         self.dropout = torch.nn.Dropout(p=0.2)
#         self.fc2 = torch.nn.Linear(32, 1)

#     def forward(self, data):
#         x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
#         for conv in self.convs:
#             x = self.dropout(F.relu(conv(x, edge_index, edge_attr)))
#         return self.fc2(F.relu(self.fc1(x))).squeeze()

# def train_dl_model(model, train_data_list, val_data_list=None, epochs=300, lr=0.01, patience=20):
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = model.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
#     all_labels = torch.cat([data.y for data in train_data_list])
#     pos_weight = ((len(all_labels) - all_labels.sum()) / (all_labels.sum() + 1e-5)).clone().detach().to(device)
#     criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
#     eval_data_list = val_data_list if val_data_list is not None else train_data_list
    
#     best_loss, epochs_no_improve = float('inf'), 0
#     best_model_wts = copy.deepcopy(model.state_dict())
    
#     for epoch in range(epochs):
#         model.train()
#         for data in train_data_list:
#             data = data.to(device)
#             optimizer.zero_grad()
#             loss = criterion(model(data), data.y)
#             loss.backward()
#             optimizer.step()
            
#         model.eval()
#         total_val_loss = 0
#         with torch.no_grad():
#             for data in eval_data_list:
#                 data = data.to(device)
#                 loss = criterion(model(data), data.y)
#                 total_val_loss += loss.item()
                
#         avg_val_loss = total_val_loss / len(eval_data_list)
        
#         if avg_val_loss < best_loss - 1e-4:
#             best_loss, epochs_no_improve, best_model_wts = avg_val_loss, 0, copy.deepcopy(model.state_dict())
#         else:
#             epochs_no_improve += 1
            
#         if (epoch + 1) % 10 == 0 or epochs_no_improve >= patience:
#             print(f"   Epoch {epoch+1:03d}/{epochs} | Val Loss: {avg_val_loss:.4f} | Best Val: {best_loss:.4f} | Patience: {epochs_no_improve}/{patience}")
            
#         if epochs_no_improve >= patience:
#             print(f"   [Early Stopping] Triggered at epoch {epoch+1}!")
#             break
            
#     print(f"   [Complete] Restored weights at Val Loss: {best_loss:.4f}")
#     model.load_state_dict(best_model_wts)
#     return model, best_loss


# # ==============================================================================
# # ALGORITHMS (BASELINES & AI-AUGMENTED) WITH ADVANCED HEURISTICS
# # ==============================================================================
# def _eval_combo_worker(combo, distances, W_scenarios, alpha, beta):
#     cost = evaluate_heuristic_robust_cost(distances, W_scenarios, list(combo), alpha, beta)
#     return cost, list(combo)

# def run_cbs(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
#     N = len(distances)
#     rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0) 
    
#     num_clusters = max(1, min(num_clusters, N // 3))
#     kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
#     cluster_labels = kmeans.fit_predict(distances) 
    
#     candidate_hubs = set()
#     for cluster_idx in range(num_clusters):
#         nodes = np.where(cluster_labels == cluster_idx)[0]
#         sorted_nodes = sorted(nodes, key=lambda x: rankings[x], reverse=True)
#         candidate_hubs.update(sorted_nodes[:3])
        
#     candidate_hubs.update(np.argsort(rankings)[-min(N, p_hubs * 2):]) 
#     candidate_hubs = list(candidate_hubs)
    
#     # [SUGGESTION C]: Dynamic Scaling (No hard cap of 16)
#     dynamic_limit = max(16, p_hubs * 5)
#     if len(candidate_hubs) > dynamic_limit:
#         candidate_hubs = sorted(candidate_hubs, key=lambda x: rankings[x], reverse=True)[:dynamic_limit]
#     if len(candidate_hubs) < p_hubs: 
#         candidate_hubs = list(np.argsort(rankings)[-p_hubs:])

#     combos = list(itertools.combinations(candidate_hubs, p_hubs))
#     results = Parallel(n_jobs=-1)(
#         delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in combos
#     )
#     if results:
#         best_cost, best_hubs = min(results, key=lambda x: x[0])
#         return best_hubs, best_cost
#     return None, float('inf')


# def run_dl_cbs(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
#     N = len(distances)
#     num_clusters = max(1, min(num_clusters, N // 2))
#     kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
#     cluster_labels = kmeans.fit_predict(distances) 
    
#     candidate_hubs = set()
#     for cluster_idx in range(num_clusters):
#         nodes = np.where(cluster_labels == cluster_idx)[0]
#         best_in_cluster = max(nodes, key=lambda x: dl_rankings[x])
#         candidate_hubs.add(best_in_cluster)
        
#     # [SUGGESTION C]: Dynamic Candidate Pool Scaling
#     dynamic_limit = max(16, p_hubs * 5)
#     sorted_ai_nodes = np.argsort(dl_rankings)[::-1]
    
#     for node in sorted_ai_nodes:
#         if len(candidate_hubs) >= dynamic_limit:
#             break
#         candidate_hubs.add(node)
        
#     candidate_hubs = list(candidate_hubs)
#     if len(candidate_hubs) < p_hubs: 
#         candidate_hubs = list(sorted_ai_nodes[:p_hubs])

#     combos = list(itertools.combinations(candidate_hubs, p_hubs))
#     results = Parallel(n_jobs=-1)(
#         delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in combos
#     )
#     if results:
#         best_cost, best_hubs = min(results, key=lambda x: x[0])
#         return best_hubs, best_cost
#     return None, float('inf')


# # [SUGGESTION A]: k-Swaps Local Search (Variable Neighborhood Descent)
# def _vnd_local_search(current_hubs, candidate_space, distances, W_scenarios, alpha, beta):
#     current_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, current_hubs, alpha, beta)
#     improved = True
    
#     while improved:
#         improved = False
        
#         # --- Neighborhood 1: 1-Swaps ---
#         best_n1_cost = current_cost
#         best_n1_hubs = current_hubs.copy()
#         for i in range(len(current_hubs)):
#             for c in candidate_space:
#                 if c not in current_hubs:
#                     temp = current_hubs.copy()
#                     temp[i] = c
#                     cost = evaluate_heuristic_robust_cost(distances, W_scenarios, temp, alpha, beta)
#                     if cost < best_n1_cost:
#                         best_n1_cost = cost
#                         best_n1_hubs = temp
                        
#         if best_n1_cost < current_cost:
#             current_cost = best_n1_cost
#             current_hubs = best_n1_hubs
#             improved = True
#             continue # Return to start of VND to perform 1-swaps again

#         # --- Neighborhood 2: 2-Swaps (Only activated if 1-swaps are stuck) ---
#         if len(current_hubs) >= 2:
#             # Truncate candidate space for 2-swaps to the top 15 to prevent computational explosion
#             short_candidates = candidate_space[:15] 
#             avail = [c for c in short_candidates if c not in current_hubs]
            
#             for i, j in itertools.combinations(range(len(current_hubs)), 2):
#                 for c1, c2 in itertools.combinations(avail, 2):
#                     temp = current_hubs.copy()
#                     temp[i], temp[j] = c1, c2
#                     cost = evaluate_heuristic_robust_cost(distances, W_scenarios, temp, alpha, beta)
#                     if cost < current_cost:
#                         current_cost = cost
#                         current_hubs = temp
#                         improved = True
#                         break # First Improvement (speed optimization)
#                 if improved:
#                     break
                    
#     return current_hubs, current_cost


# # [SUGGESTION B]: Dynamic Shaking Intensity ($k_{max}$)
# def _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter, epsilon_greedy=False, all_nodes_space=None):
#     best_hubs = initial_hubs.copy()
#     best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    
#     for _ in range(max_iter):
#         k_shake = 1
#         k_max = max(1, len(best_hubs) - 1) # Shake up to p-1 hubs simultaneously
        
#         while k_shake <= k_max:
#             test_hubs = best_hubs.copy()
#             shake_indices = np.random.choice(len(test_hubs), k_shake, replace=False)
            
#             # Epsilon-Greedy logic for DL-GVNS
#             if epsilon_greedy and np.random.rand() < 0.20 and all_nodes_space is not None:
#                 avail = [c for c in all_nodes_space if c not in test_hubs]
#             else:
#                 avail = [c for c in candidate_space if c not in test_hubs]
                
#             if len(avail) >= k_shake:
#                 new_hubs = np.random.choice(avail, k_shake, replace=False)
#                 for idx, new_hub in zip(shake_indices, new_hubs):
#                     test_hubs[idx] = new_hub
                    
#             # Engage the robust Variable Neighborhood Descent (k-Swaps)
#             test_hubs, test_cost = _vnd_local_search(test_hubs, candidate_space, distances, W_scenarios, alpha, beta)
            
#             if test_cost < best_cost:
#                 best_cost = test_cost
#                 best_hubs = test_hubs
#                 k_shake = 1 # Return to lowest shake intensity on success
#             else:
#                 k_shake += 1 # Increase intensity if stuck
                
#     return best_hubs, best_cost


# def run_gvns(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, max_iter=30):
#     N = len(distances)
#     naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0)
#     candidate_space = list(np.argsort(naive_rankings)[-min(N, 40):])
#     current_hubs = list(np.random.choice(candidate_space, p_hubs, replace=False))
    
#     return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)


# def run_dl_gvns(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, max_iter=30):
#     N = len(distances)
#     sorted_ai_nodes = np.argsort(dl_rankings)[::-1]
#     all_nodes_space = list(range(N))
    
#     guaranteed_p = min(p_hubs, max(1, p_hubs // 2))
#     initial_hubs = list(sorted_ai_nodes[:guaranteed_p])
#     remaining_p = p_hubs - guaranteed_p
    
#     wider_pool = list(sorted_ai_nodes[guaranteed_p:min(N, 40)])
#     if remaining_p > 0:
#         initial_hubs.extend(np.random.choice(wider_pool, remaining_p, replace=False))
        
#     candidate_space = list(sorted_ai_nodes[:min(N, 40)])
    
#     return _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter, epsilon_greedy=True, all_nodes_space=all_nodes_space)


# # ==============================================================================
# # PHASE 4: RESULTS EXPORT & PLOTTING
# # ==============================================================================
# def plot_academic_results(df):
#     print("\n[Plotting] Generating Academic Subplot Charts...")
#     sns.set_theme(style="whitegrid")
    
#     g1 = sns.catplot(data=df, x='Hubs', y='Robust Cost', hue='Method', col='Nodes', 
#                      kind='bar', sharey=False, col_wrap=2, height=4, aspect=1.2, palette="viridis")
#     g1.fig.suptitle("Robust Cost Comparison Across Methods by Dataset Size", y=1.02)
#     for ax in g1.axes.flat:
#         ax.set_yscale('log')
#     g1.savefig("Fig1_Cost_Comparison_Subplots.png", bbox_inches='tight', dpi=300)
#     plt.close()

#     g2 = sns.catplot(data=df, x='Hubs', y='Time (s)', hue='Method', col='Nodes', 
#                      kind='bar', sharey=False, col_wrap=2, height=4, aspect=1.2, palette="viridis")
#     g2.fig.suptitle("Execution Time by Method and Dataset Size", y=1.02)
#     for ax in g2.axes.flat:
#         ax.set_yscale('log')
#     g2.savefig("Fig2_Time_Comparison_Subplots.png", bbox_inches='tight', dpi=300)
#     plt.close()

#     df_heuristics = df[df['Method'] != 'Exact (SCIP)']
#     plt.figure(figsize=(10, 6))
#     sns.lineplot(data=df_heuristics, x='Nodes', y='Gap (%)', hue='Method', marker='o', linewidth=2.5)
#     plt.title("Optimality Gap Scaling with Network Size")
#     plt.ylabel("Gap from Exact Solver (%)")
#     plt.axhline(0, color='black', linestyle='--')
#     plt.tight_layout()
#     plt.savefig("Fig3_Gap_Scaling.png", dpi=300)
#     plt.close()
    
#     print("[Plotting] Charts saved: Fig1_Cost_Comparison_Subplots.png, Fig2_Time_Comparison_Subplots.png, Fig3_Gap_Scaling.png")


# # ==============================================================================
# # MASTER PIPELINE EXECUTION
# # ==============================================================================
# if __name__ == "__main__":
#     print("==================================================")
#     print("🚀 STARTING HYBRID AI-OR ACADEMIC PIPELINE")
#     print("==================================================")
    
#     DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
#     distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
#     if distance_matrices and demand_matrices:
#         common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
        
#         # --- PHASE 1: LOAD OFFLINE TRAINING DATA ---
#         dataset_path = "/kaggle/input/datasets/infernalss/gnn-training/offline_training_dataset.pt"
#         training_graphs = []
        
#         print("\n[Phase 1] Loading pre-generated offline training dataset...")
#         if os.path.exists(dataset_path):
#             # weights_only=False bypasses the strict PyTorch 2.6 security check for complex geometric data
#             training_graphs = torch.load(dataset_path, weights_only=False)
#             print(f"   [✓] Successfully loaded {len(training_graphs)} graphs from '{dataset_path}'.")
#         else:
#             print(f"   [!] Error: '{dataset_path}' not found!")
#             print("       Please run your dataset generator script first to build the data.")
            
#         # --- PHASE 2: DEEP HYPERPARAMETER SEARCH (GNN) ---
#         print("\n\n" + "="*50)
#         print("PHASE 2: DEEP GNN HYPERPARAMETER SEARCH")
#         print("="*50)
        
#         device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#         best_model = None
#         best_val_loss = float('inf')
#         best_hidden_dim = None
#         best_num_layers = None
        
#         if len(training_graphs) > 4:
#             random.shuffle(training_graphs)
#             split_idx = int(len(training_graphs) * 0.8)
#             train_set = training_graphs[:split_idx]
#             val_set = training_graphs[split_idx:]
            
#             # [DEEP SEARCH]: Testing combinations of Width AND Depth
#             hidden_dims_to_test = [16]
#             layers_to_test = [2] 
            
#             print(f"Train Set: {len(train_set)} graphs | Validation Set: {len(val_set)} graphs")
            
#             for h_dim in hidden_dims_to_test:
#                 for n_layers in layers_to_test:
#                     print(f"\n--- Testing Architecture: Hidden Dim = {h_dim} | Layers = {n_layers} ---")
#                     temp_model = MultiGraphHubRanker(node_in_dim=3, edge_in_dim=3, hidden_dim=h_dim, num_layers=n_layers)
                    
#                     trained_model, final_val_loss = train_dl_model(
#                         temp_model, train_data_list=train_set, val_data_list=val_set, 
#                         epochs=300, patience=20
#                     )
                    
#                     if final_val_loss < best_val_loss:
#                         best_val_loss = final_val_loss
#                         best_model = trained_model
#                         best_hidden_dim = h_dim
#                         best_num_layers = n_layers
                        
#             print(f"\n🌟 [Hyperparameter Search Complete]")
#             print(f"   Winning Architecture: {best_hidden_dim} Neurons, {best_num_layers} Layers (Val Loss: {best_val_loss:.4f})")
#             dl_model = best_model
#             dl_model.eval()
#         else:
#             print("[!] Not enough training data generated. Skipping Phase 2.")
#             dl_model = MultiGraphHubRanker()
        
#         # --- PHASE 3: PARAMETER GRID SEARCH ---
#         print("\n\n" + "="*50)
#         print("PHASE 3: GRID SEARCH & HYBRID BASELINE EVALUATIONS")
#         print("="*50)
        
#         results_list = []
#         p_values = [2, 3, 4, 5] 
        
#         for N in common_nodes:
#             distances, demands = distance_matrices[N], demand_matrices[N]
#             W_scenarios = generate_demand_scenarios(demands, num_scenarios=100)
            
#             with torch.no_grad():
#                 graph_data = create_multi_graph_data(distances, demands).to(device)
#                 ai_probs = torch.sigmoid(dl_model(graph_data)).cpu().numpy()
            
#             for p in p_values:
#                 print(f"\n--- Evaluating Nodes={N} | Hubs(p)={p} ---")
                
#                 # 1. Exact Solver
#                 start_t = time.time()
#                 try:
#                     exact_hubs, exact_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=1.5)
#                 except:
#                     exact_cost = float('inf')
#                 t_exact = time.time() - start_t
                
#                 if exact_cost != float('inf'):
#                     results_list.append({'Nodes': N, 'Hubs': p, 'Method': 'Exact (SCIP)', 'Time (s)': t_exact, 'Robust Cost': exact_cost, 'Gap (%)': 0.0})
#                     print(f"   [Exact] Time: {t_exact:.2f}s | Cost: {exact_cost:.2e}")
                
#                 def run_and_log(method_name, func, *args):
#                     start_t = time.time()
#                     hubs, cost = func(*args)
                    
#                     if exact_cost != float('inf'):
#                         _, true_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=0.5, fixed_hubs=hubs)
#                         gap = ((true_cost - exact_cost) / exact_cost) * 100
#                     else:
#                         true_cost, gap = cost, 0.0
                        
#                     t_val = time.time() - start_t
#                     results_list.append({'Nodes': N, 'Hubs': p, 'Method': method_name, 'Time (s)': t_val, 'Robust Cost': true_cost, 'Gap (%)': gap})
#                     print(f"   [{method_name}] Time: {t_val:.2f}s | Gap: {gap:.2f}% | Hubs: {hubs}")

#                 # 2. Heuristics Evaluation
#                 run_and_log('CBS (Standard)', run_cbs, distances, W_scenarios, demands, p)
#                 run_and_log('DL-CBS (AI)', run_dl_cbs, distances, W_scenarios, ai_probs, p)
#                 run_and_log('GVNS (Standard)', run_gvns, distances, W_scenarios, demands, p)
#                 run_and_log('DL-GVNS (AI)', run_dl_gvns, distances, W_scenarios, ai_probs, p)

#         # --- PHASE 4: EXPORT & PLOT ---
#         if results_list:
#             df_results = pd.DataFrame(results_list)
#             df_results.to_csv("All_Datasets_Results.csv", index=False)
#             plot_academic_results(df_results)
            
#             print("\n==================================================")
#             print("✅ BATCH PIPELINE COMPLETE!")
#             print("📈 All plots saved as PNG images.")
#             print("💾 Master data saved to 'All_Datasets_Results.csv'.")
#             print("==================================================")

#     else:
#         print("\n[!] Directory not found or empty.")

In [4]:
import os
import re
import time
import copy
import random
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from ortools.linear_solver import pywraplp
from joblib import Parallel, delayed

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv

# ==============================================================================
# 1. DATASET LOADING & PREPROCESSING
# ==============================================================================
def load_all_datasets(folder_path):
    distance_matrices = {}
    demand_matrices = {}
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None, None

    print(f"Scanning directory: {folder_path}...\n")
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if not os.path.isfile(file_path): continue
            
        match = re.search(r'\d+', filename)
        if not match: continue
        nodes = int(match.group())
        
        try:
            if filename.startswith(('C-', 'C_')) and filename.endswith('.csv'):
                df = pd.read_csv(file_path, header=None)
                distance_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Distance Matrix (C) for {nodes} nodes.")
                
            elif filename.startswith('wij'):
                if filename.endswith('.xlsx'): df = pd.read_excel(file_path)
                elif filename.endswith('.csv'): df = pd.read_csv(file_path)
                else: continue
                
                if df.shape[1] > nodes: df = df.iloc[:, 1:]
                demand_matrices[nodes] = df.values
                print(f"[\u2713] Loaded Demand Matrix (W) for {nodes} nodes.")
        except Exception as e:
            pass
    return distance_matrices, demand_matrices

def calculate_node_production_attraction(W_matrix):
    return np.sum(W_matrix, axis=1), np.sum(W_matrix, axis=0)

def generate_demand_scenarios(W_matrix, num_scenarios=100, variance_factor=0.2, seed=42):
    np.random.seed(seed)
    n = W_matrix.shape[0]
    W_scenarios = np.random.normal(loc=W_matrix, scale=W_matrix * variance_factor, size=(num_scenarios, n, n))
    W_scenarios = np.maximum(W_scenarios, 0)
    return np.transpose(W_scenarios, (1, 2, 0))

# ==============================================================================
# 2. EXACT SOLVER & MATHEMATICAL EVALUATION
# ==============================================================================
def solve_exact_robust_model(distances, W_scenarios, p_hubs, alpha=0.5, beta=0.9, time_limit_mins=1.5, fixed_hubs=None):
    N = len(distances)
    num_scenarios = W_scenarios.shape[2]
    
    dist_scale = np.max(distances) if np.max(distances) > 0 else 1.0
    w_scale = np.max(W_scenarios) if np.max(W_scenarios) > 0 else 1.0
    scaled_distances, scaled_W = distances / dist_scale, W_scenarios / w_scale
    cost_multiplier = dist_scale * w_scale 
    
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if not solver: return None, float('inf')
    solver.SetTimeLimit(int(time_limit_mins * 60 * 1000))

    y = {k: solver.IntVar(0, 1, f'y_{k}') for k in range(N)}
    z = {(i, k): solver.IntVar(0, 1, f'z_{i}_{k}') for k in range(N) for i in range(N)}
    mu = solver.NumVar(0, solver.infinity(), 'mu')
    lambda_s = {s: solver.NumVar(0, solver.infinity(), f'lambda_{s}') for s in range(num_scenarios)}

    solver.Add(solver.Sum([y[k] for k in range(N)]) == p_hubs)

    # Force exact evaluation of heuristic hubs for perfect gap calculation
    if fixed_hubs is not None:
        for k in range(N):
            if k in fixed_hubs:
                solver.Add(y[k] == 1)
            else:
                solver.Add(y[k] == 0)

    for i in range(N):
        solver.Add(solver.Sum([z[i, k] for k in range(N)]) == 1)
        for k in range(N):
            solver.Add(z[i, k] <= y[k])

    for s in range(num_scenarios):
        W_s = scaled_W[:, :, s]
        cost_s_expr = 0
        for i in range(N):
            for k in range(N):
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[j, i] for j in range(N)) * scaled_distances[i, k] * z[i, k]
                cost_s_expr += sum(W_s[i, j] for j in range(N)) * (alpha * scaled_distances[k, k]) * z[i, k]
        solver.Add(lambda_s[s] >= cost_s_expr - mu)

    penalty_weight = 1.0 / (num_scenarios * (1.0 - beta))
    solver.Minimize(mu + penalty_weight * solver.Sum([lambda_s[s] for s in range(num_scenarios)]))

    status = solver.Solve()
    if status in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        return [k for k in range(N) if y[k].solution_value() > 0.5], solver.Objective().Value() * cost_multiplier
    return None, float('inf')

def evaluate_heuristic_robust_cost(distances, W_scenarios, hubs, alpha=0.5, beta=0.9):
    N, num_scenarios = len(distances), W_scenarios.shape[2]
    allocation = {i: min(hubs, key=lambda h: distances[i, h]) for i in range(N)}
    
    path_costs = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            hi, hj = allocation[i], allocation[j]
            path_costs[i, j] = distances[i, hi] + alpha * distances[hi, hj] + distances[hj, j]
            
    scenario_costs = np.tensordot(path_costs, W_scenarios, axes=([0, 1], [0, 1]))
    sorted_costs = np.sort(scenario_costs)
    var_index = int(beta * num_scenarios)
    mu = sorted_costs[var_index]
    
    return mu + np.sum(sorted_costs[var_index:] - mu) / (num_scenarios * (1.0 - beta))

# ==============================================================================
# 3. GRAPH NEURAL NETWORK
# ==============================================================================
def create_multi_graph_data(distances, demands, labels=None):
    N = len(distances)
    production, attraction = calculate_node_production_attraction(demands)
    distance_sums = np.sum(distances, axis=1)
    spatial_centrality = 1.0 / (distance_sums + 1e-6)
    
    x = torch.tensor(np.column_stack((
        production / (np.max(production) or 1.0), 
        attraction / (np.max(attraction) or 1.0),
        spatial_centrality / (np.max(spatial_centrality) or 1.0)
    )), dtype=torch.float)

    edge_index, edge_attr = [], []
    dist_max, dem_max = (np.max(distances) or 1.0), (np.max(demands) or 1.0)
    flow = demands + demands.T
    flow_max = np.max(flow) or 1.0

    for i in range(N):
        for j in range(N):
            if i != j:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]/dist_max, demands[i, j]/dem_max, flow[i, j]/flow_max])

    data = Data(x=x, 
                edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(), 
                edge_attr=torch.tensor(edge_attr, dtype=torch.float))
    if labels is not None: data.y = torch.tensor(labels, dtype=torch.float)
    return data

class MultiGraphHubRanker(torch.nn.Module):
    def __init__(self, node_in_dim=3, edge_in_dim=3, hidden_dim=64, num_layers=2):
        super(MultiGraphHubRanker, self).__init__()
        self.num_layers = num_layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(TransformerConv(node_in_dim, hidden_dim, edge_dim=edge_in_dim))
        for _ in range(num_layers - 1):
            self.convs.append(TransformerConv(hidden_dim, hidden_dim, edge_dim=edge_in_dim))
        self.fc1 = torch.nn.Linear(hidden_dim, 32)
        self.dropout = torch.nn.Dropout(p=0.2)
        self.fc2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        for conv in self.convs:
            x = self.dropout(F.relu(conv(x, edge_index, edge_attr)))
        return self.fc2(F.relu(self.fc1(x))).squeeze()

def train_dl_model(model, train_data_list, val_data_list=None, epochs=300, lr=0.01, patience=20):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    all_labels = torch.cat([data.y for data in train_data_list])
    pos_weight = ((len(all_labels) - all_labels.sum()) / (all_labels.sum() + 1e-5)).clone().detach().to(device)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    eval_data_list = val_data_list if val_data_list is not None else train_data_list
    best_loss, epochs_no_improve = float('inf'), 0
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        model.train()
        for data in train_data_list:
            data = data.to(device)
            optimizer.zero_grad()
            loss = criterion(model(data), data.y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in eval_data_list:
                data = data.to(device)
                loss = criterion(model(data), data.y)
                total_val_loss += loss.item()
                
        avg_val_loss = total_val_loss / len(eval_data_list)
        
        if avg_val_loss < best_loss - 1e-4:
            best_loss, epochs_no_improve, best_model_wts = avg_val_loss, 0, copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            
        if (epoch + 1) % 10 == 0 or epochs_no_improve >= patience:
            print(f"   Epoch {epoch+1:03d}/{epochs} | Val Loss: {avg_val_loss:.4f} | Best Val: {best_loss:.4f} | Patience: {epochs_no_improve}/{patience}")
            
        if epochs_no_improve >= patience:
            break
            
    model.load_state_dict(best_model_wts)
    return model, best_loss


# ==============================================================================
# 4. ALGORITHMS (BASELINES & AI-AUGMENTED)
# ==============================================================================
def _eval_combo_worker(combo, distances, W_scenarios, alpha, beta):
    cost = evaluate_heuristic_robust_cost(distances, W_scenarios, list(combo), alpha, beta)
    return cost, list(combo)

def run_cbs(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    """STANDARD CBS: Uses Naive Demand Volume for ranking, no AI."""
    N = len(distances)
    naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0) 
    return _cbs_core(distances, W_scenarios, naive_rankings, p_hubs, alpha, beta, num_clusters)

def run_dl_cbs(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, num_clusters=5):
    """DL-CBS: Uses AI Probabilities for ranking."""
    return _cbs_core(distances, W_scenarios, dl_rankings, p_hubs, alpha, beta, num_clusters)

def _cbs_core(distances, W_scenarios, rankings, p_hubs, alpha, beta, num_clusters, top_c=3):
    N = len(distances)
    num_clusters = max(1, min(num_clusters, N // top_c))
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(distances) 
    
    candidate_hubs = set()
    for cluster_idx in range(num_clusters):
        nodes = np.where(cluster_labels == cluster_idx)[0]
        sorted_nodes = sorted(nodes, key=lambda x: rankings[x], reverse=True)
        candidate_hubs.update(sorted_nodes[:top_c])
        
    candidate_hubs.update(np.argsort(rankings)[-min(N, p_hubs * 2):]) 
    candidate_hubs = list(candidate_hubs)
    
    # Hard Cap for Speed
    if len(candidate_hubs) > 16:
        candidate_hubs = sorted(candidate_hubs, key=lambda x: rankings[x], reverse=True)[:16]
        
    if len(candidate_hubs) < p_hubs: 
        candidate_hubs = list(np.argsort(rankings)[-p_hubs:])

    combos = list(itertools.combinations(candidate_hubs, p_hubs))
    results = Parallel(n_jobs=-1)(
        delayed(_eval_combo_worker)(c, distances, W_scenarios, alpha, beta) for c in combos
    )
    if results:
        best_cost, best_hubs = min(results, key=lambda x: x[0])
        return best_hubs, best_cost
    return None, float('inf')

def run_gvns(distances, W_scenarios, demands, p_hubs, alpha=0.5, beta=0.9, max_iter=50):
    """STANDARD GVNS: Variable Neighborhood Search"""
    N = len(distances)
    naive_rankings = np.sum(demands, axis=1) + np.sum(demands, axis=0)
    candidate_space = np.argsort(naive_rankings)[-min(N, 40):] 
    current_hubs = list(np.random.choice(candidate_space, p_hubs, replace=False))
    return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)

def run_dl_gvns(distances, W_scenarios, dl_rankings, p_hubs, alpha=0.5, beta=0.9, max_iter=50):
    """DL-GVNS: AI-Initialized, AI-Restricted swapping neighborhood."""
    N = len(distances)
    current_hubs = list(np.argsort(dl_rankings)[-p_hubs:])
    candidate_space = np.argsort(dl_rankings)[-min(N, 40):]
    return _gvns_core(distances, W_scenarios, current_hubs, candidate_space, alpha, beta, max_iter)

def _gvns_core(distances, W_scenarios, initial_hubs, candidate_space, alpha, beta, max_iter):
    best_hubs = initial_hubs.copy()
    best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    
    for _ in range(max_iter):
        improved = True
        while improved:
            improved = False
            for i in range(len(best_hubs)):
                for c in candidate_space:
                    if c not in best_hubs:
                        test_hubs = best_hubs.copy()
                        test_hubs[i] = c
                        cost = evaluate_heuristic_robust_cost(distances, W_scenarios, test_hubs, alpha, beta)
                        if cost < best_cost:
                            best_cost, best_hubs, improved = cost, test_hubs, True
                            
        shake_idx = np.random.randint(0, len(best_hubs))
        shake_candidate = np.random.choice([c for c in candidate_space if c not in best_hubs])
        best_hubs[shake_idx] = shake_candidate
        
    best_cost = evaluate_heuristic_robust_cost(distances, W_scenarios, best_hubs, alpha, beta)
    return best_hubs, best_cost


# ==============================================================================
# 5. DATA PLOTTING ENGINE
# ==============================================================================
def plot_academic_results(dataframe):
    print("\n[Plotting] Generating Academic Subplot Charts for your Thesis...")
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    
    # 1. Cost Comparison Subplots
    g1 = sns.catplot(data=dataframe, x='Hubs', y='Robust Cost', hue='Method', col='Nodes', 
                     kind='bar', sharey=False, col_wrap=3, height=4, aspect=1.2, palette="viridis")
    g1.fig.suptitle("Robust Cost Comparison Across Methods by Dataset Size", y=1.05, fontweight="bold")
    for ax in g1.axes.flat:
        ax.set_yscale('log')
        ax.set_ylabel("Conditional Value at Risk (Cost)")
    g1.savefig("Fig1_Cost_Comparison_Complete.png", bbox_inches='tight', dpi=300)
    plt.close()

    # 2. Execution Time Subplots
    g2 = sns.catplot(data=dataframe, x='Hubs', y='Time (s)', hue='Method', col='Nodes', 
                     kind='bar', sharey=False, col_wrap=3, height=4, aspect=1.2, palette="viridis")
    g2.fig.suptitle("Execution Time by Method and Dataset Size", y=1.05, fontweight="bold")
    for ax in g2.axes.flat:
        ax.set_yscale('log')
        ax.set_ylabel("Time (seconds)")
    g2.savefig("Fig2_Time_Comparison_Complete.png", bbox_inches='tight', dpi=300)
    plt.close()

    # 3. Gap Scaling Line Chart
    df_heuristics = dataframe[dataframe['Method'] != 'Exact (SCIP)']
    plt.figure(figsize=(10, 6))
    df_agg = df_heuristics.groupby(['Nodes', 'Method'])['Gap (%)'].mean().reset_index()
    sns.lineplot(data=df_agg, x='Nodes', y='Gap (%)', hue='Method', marker='o', linewidth=3, markersize=10, palette="viridis")
    
    plt.title("Average Optimality Gap Scaling with Network Size", fontweight="bold")
    plt.ylabel("Average Gap from Exact Solver (%)")
    plt.xlabel("Number of Nodes (N)")
    plt.axhline(0, color='black', linestyle='--', linewidth=1.5)
    plt.tight_layout()
    plt.savefig("Fig3_Gap_Scaling_Complete.png", dpi=300)
    plt.close()
    
    print("📈 Plots successfully saved!")


# ==============================================================================
# 6. MASTER PIPELINE EXECUTION (N=200 ONLY + MERGE)
# ==============================================================================
if __name__ == "__main__":
    print("==================================================")
    print("🚀 STARTING N=200 HYBRID AI-OR EVALUATION")
    print("==================================================")
    
    DATA_FOLDER = "/kaggle/input/datasets/infernalss/node-dataset" 
    distance_matrices, demand_matrices = load_all_datasets(DATA_FOLDER)
    
    if distance_matrices and demand_matrices:
        common_nodes = sorted(list(set(distance_matrices.keys()) & set(demand_matrices.keys())))
        
        # --- PHASE 1: LOAD OFFLINE TRAINING DATA ---
        dataset_path = "/kaggle/input/datasets/infernalss/gnn-training/offline_training_dataset.pt"
        training_graphs = []
        
        print("\n[Phase 1] Loading pre-generated offline training dataset...")
        if os.path.exists(dataset_path):
            training_graphs = torch.load(dataset_path, weights_only=False)
            print(f"   [✓] Successfully loaded {len(training_graphs)} graphs from '{dataset_path}'.")
        else:
            print(f"   [!] '{dataset_path}' not found! The GNN will initialize with random weights if skipped.")
            
        # --- PHASE 2: DEEP HYPERPARAMETER SEARCH (GNN) ---
        print("\n\n" + "="*50)
        print("PHASE 2: DEEP GNN HYPERPARAMETER SEARCH")
        print("="*50)
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        best_model = None
        best_val_loss = float('inf')
        best_hidden_dim = None
        best_num_layers = None
        
        if len(training_graphs) > 4:
            random.shuffle(training_graphs)
            split_idx = int(len(training_graphs) * 0.8)
            train_set = training_graphs[:split_idx]
            val_set = training_graphs[split_idx:]
            
            hidden_dims_to_test = [16]
            layers_to_test = [2] 
            
            for h_dim in hidden_dims_to_test:
                for n_layers in layers_to_test:
                    temp_model = MultiGraphHubRanker(node_in_dim=3, edge_in_dim=3, hidden_dim=h_dim, num_layers=n_layers)
                    trained_model, final_val_loss = train_dl_model(
                        temp_model, train_data_list=train_set, val_data_list=val_set, epochs=300, patience=20
                    )
                    if final_val_loss < best_val_loss:
                        best_val_loss = final_val_loss
                        best_model = trained_model
                        best_hidden_dim = h_dim
                        best_num_layers = n_layers
                        
            print(f"\n🌟 [Hyperparameter Search Complete] Winning Architecture: {best_hidden_dim} Neurons, {best_num_layers} Layers")
            dl_model = best_model
            dl_model.eval()
        else:
            print("[!] Skipping Phase 2.")
            dl_model = MultiGraphHubRanker()
        
        # --- PHASE 3: PARAMETER GRID SEARCH (RESTRICTED TO N=200) ---
        print("\n\n" + "="*50)
        print("PHASE 3: GRID SEARCH & HYBRID BASELINE EVALUATIONS (N=200 ONLY)")
        print("="*50)
        
        results_list = []
        p_values = [2, 3, 4, 5] 
        
        # ---> HARD FILTER: ONLY EVALUATE 200 NODE <---
        nodes_to_evaluate = [n for n in common_nodes if n == 200]
        
        if not nodes_to_evaluate:
            print("\n[!] Error: Dataset for N=200 was not found in your directory!")
        else:
            for N in nodes_to_evaluate:
                distances, demands = distance_matrices[N], demand_matrices[N]
                W_scenarios = generate_demand_scenarios(demands, num_scenarios=100)
                
                with torch.no_grad():
                    graph_data = create_multi_graph_data(distances, demands).to(device)
                    ai_probs = torch.sigmoid(dl_model(graph_data)).cpu().numpy()
                
                for p in p_values:
                    print(f"\n--- Evaluating Nodes={N} | Hubs(p)={p} ---")
                    
                    start_t = time.time()
                    try:
                        exact_hubs, exact_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=1.5)
                    except:
                        exact_cost = float('inf')
                    t_exact = time.time() - start_t
                    
                    if exact_cost != float('inf'):
                        results_list.append({'Nodes': N, 'Hubs': p, 'Method': 'Exact (SCIP)', 'Time (s)': t_exact, 'Robust Cost': exact_cost, 'Gap (%)': 0.0})
                        print(f"   [Exact] Time: {t_exact:.2f}s | Cost: {exact_cost:.2e}")
                    
                    def run_and_log(method_name, func, *args):
                        start_t = time.time()
                        hubs, cost = func(*args)
                        
                        if exact_cost != float('inf'):
                            # Forces SCIP solver to calculate True Gap using heuristic's selected hubs
                            _, true_cost = solve_exact_robust_model(distances, W_scenarios, p, time_limit_mins=0.5, fixed_hubs=hubs)
                            gap = ((true_cost - exact_cost) / exact_cost) * 100
                        else:
                            true_cost, gap = cost, 0.0
                            
                        t_val = time.time() - start_t
                        results_list.append({'Nodes': N, 'Hubs': p, 'Method': method_name, 'Time (s)': t_val, 'Robust Cost': true_cost, 'Gap (%)': gap})
                        print(f"   [{method_name}] Time: {t_val:.2f}s | Gap: {gap:.2f}% | Hubs: {hubs}")

                    run_and_log('CBS (Standard)', run_cbs, distances, W_scenarios, demands, p)
                    run_and_log('DL-CBS (AI)', run_dl_cbs, distances, W_scenarios, ai_probs, p)
                    run_and_log('GVNS (Standard)', run_gvns, distances, W_scenarios, demands, p)
                    run_and_log('DL-GVNS (AI)', run_dl_gvns, distances, W_scenarios, ai_probs, p)

            # --- PHASE 4: MERGE WITH N150 CSV AND PLOT ---
            print("\n\n" + "="*50)
            print("PHASE 4: DATA MERGING AND ACADEMIC PLOTTING")
            print("="*50)
            
            if results_list:
                df_n200 = pd.DataFrame(results_list)
                
                # Automatically stacks with the N150 file if it's placed in the Kaggle working directory
                csv_past = "/kaggle/input/datasets/infernalss/kaggle-results-up-to-n150/Kaggle_Results_Up_To_N150.csv"
                if os.path.exists(csv_past):
                    print(f"\n[Data Merge] Found '{csv_past}'! Merging N=200 results with previous data...")
                    df_past = pd.read_csv(csv_past)
                    df_all = pd.concat([df_past, df_n200], ignore_index=True)
                else:
                    print(f"\n[Data Merge] '{csv_past}' not found. Generating plots for N=200 data only.")
                    df_all = df_n200
                    
                final_csv_name = "All_Datasets_Results_Complete.csv"
                df_all.to_csv(final_csv_name, index=False)
                print(f"💾 Saved complete master dataset to: '{final_csv_name}'")
                
                # Will draw subplots for 25, 100, 150, AND 200 side-by-side
                plot_academic_results(df_all)
                
                print("\n==================================================")
                print("✅ MASTER PIPELINE COMPLETE!")
                print("==================================================")

    else:
        print("\n[!] Directory not found or empty.")

🚀 STARTING N=200 HYBRID AI-OR EVALUATION
Scanning directory: /kaggle/input/datasets/infernalss/node-dataset...

[✓] Loaded Demand Matrix (W) for 100 nodes.
[✓] Loaded Distance Matrix (C) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 200 nodes.
[✓] Loaded Demand Matrix (W) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 150 nodes.
[✓] Loaded Distance Matrix (C) for 100 nodes.
[✓] Loaded Demand Matrix (W) for 25 nodes.
[✓] Loaded Distance Matrix (C) for 25 nodes.

[Phase 1] Loading pre-generated offline training dataset...
   [✓] Successfully loaded 560 graphs from '/kaggle/input/datasets/infernalss/gnn-training/offline_training_dataset.pt'.


PHASE 2: DEEP GNN HYPERPARAMETER SEARCH
   Epoch 010/300 | Val Loss: 1.0623 | Best Val: 1.0606 | Patience: 2/20
   Epoch 020/300 | Val Loss: 1.0560 | Best Val: 0.9956 | Patience: 2/20
   Epoch 030/300 | Val Loss: 0.9801 | Best Val: 0.9754 | Patience: 4/20
   Epoch 040/300 | Val Loss: 0.9834 | Best Val: 0.9549 | Patience: 1/20
   Epoch 050/300 |